In [1]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [3]:
from starter import rag

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

It keeps calling the model in a `while True` loop.

Each iteration:
1. Send the full message history to the model.
2. Check the response for any `function_call` items.
3. If there are function calls, run the tools, append the results, and loop again.
4. If there are no function calls, `break`.

So the stop condition is: **the model returns a response with no function calls**.


In [ ]:
class RAGTraced(RAGBase):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.tracer = trace.get_tracer("llm-zoomcamp")

    def rag(self, query: str) -> str:
        with self.tracer.start_as_current_span("rag"):
            return super().rag(query)
        
    def search(self, query: str, num_results: int = 5) -> list:
        with self.tracer.start_as_current_span("search"):
            return super().search(query, num_results=num_results)

    def llm(self, prompt: str) -> str:
        with self.tracer.start_as_current_span("llm"):
            return super().llm(prompt)